In [1]:
import pandas as pd
import mysql.connector

In [2]:
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="banumathi@#$0",
    database="revenue_prediction")
print('conect success')

conect success


In [3]:
query="""
WITH order_analysis AS (
    SELECT
        so.order_id,
        so.order_date,
        so.product,
        so.region,
        so.salesperson_id,
        sp.salesperson_name,
        so.list_price,
        so.approved_discount_pct,
        dp.max_discount_pct,
        i.invoice_price,

        -- Allowed price based on policy
        ROUND(
            so.list_price * (1 - dp.max_discount_pct / 100),
            2
        ) AS allowed_price,

        -- Revenue leakage calculation
        ROUND(
            (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price,
            2
        ) AS revenue_leakage,

        -- Policy violation flag
        CASE
            WHEN so.approved_discount_pct > dp.max_discount_pct
            THEN 'YES'
            ELSE 'NO'
        END AS policy_violation

    FROM sales_orders so
    JOIN invoices i
        ON so.order_id = i.order_id
    JOIN salespersons sp
        ON so.salesperson_id = sp.salesperson_id
    JOIN discount_policy dp
        ON so.product = dp.product
)

SELECT *
FROM order_analysis
WHERE revenue_leakage > 0;
"""

In [4]:
df = pd.read_sql(query, conn)
df.to_csv("revenue_leakage_orders.csv", index=False)


C:\Users\ELCOT\AppData\Local\Temp\ipykernel_12188\3246997312.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [5]:
df.head()

,order_id,order_date,product,region,salesperson_id,salesperson_name,list_price,approved_discount_pct,max_discount_pct,invoice_price,allowed_price,revenue_leakage,policy_violation
0,ORD0001,2024-08-05,Laptop,North,SP012,Salesperson_12,33843,19,16,20542,28428.12,7886.12,YES
1,ORD0004,2024-04-13,Printer,East,SP008,Salesperson_8,58784,12,19,18702,47615.04,28913.04,NO
2,ORD0005,2024-04-14,Printer,North,SP001,Salesperson_1,47835,17,19,17241,38746.35,21505.35,NO
3,ORD0007,2024-06-27,Tablet,West,SP006,Salesperson_6,66056,12,15,18916,56147.60,37231.60,NO
4,ORD0008,2024-04-20,Laptop,North,SP013,Salesperson_13,77413,13,16,45701,65026.92,19325.92,NO


## 📌 Dataset Name

Order-Level Revenue Leakage Dataset

## 🎯 Business Problem

The company was losing revenue due to discounts exceeding policy limits and inconsistent invoice pricing.

## 🧠 What I Did

Integrated sales, invoice, salesperson, and policy data

Calculated allowed price vs actual invoice price

Identified revenue leakage per order

Flagged policy violations


## 🛠 SQL Skills Used

JOINs (multi-table integration)

CTE (clean business logic)

CASE WHEN (policy flag)

Calculated columns

Business filtering


## 📈 Business Impact

Identified loss-making orders

Enabled management to audit risky sales

Foundation dataset for dashboards & decisions